In [1]:

import os
import numpy as np
import json
from openai import OpenAI
from pathlib import Path
from sentence_transformers import SentenceTransformer
from typing import Tuple

# Initialize the OpenAI client with the base URL and API key
client = OpenAI(
    base_url="https://api.studio.nebius.com/v1/",
    api_key=os.getenv(
        "OPENAI_API_KEY"
    ),  # Retrieve the API key from environment variables
)
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")


def extract_json_from(json_file_path: Path) -> list[str]:
    # 폴더 경로 내 모든 json 파일을 읽어 데이터를 추출
    with open(json_file_path, "r") as f:
        nl_task_dict = json.load(f)
    return nl_task_dict


def embed_data(data: list[str], model: SentenceTransformer) -> list[list[float]]:
    # 데이터 임베딩
    return model.encode(data)


def compute_cosine_similarity(
    vec1: list[list[float]], vec2: list[list[float]]
) -> float:
    # 코사인 유사도 계산
    return np.dot(vec1, vec2) / (np.linalg.norm(vec1) * np.linalg.norm(vec2))


def semantic_search(
    query_vec: list[list[float]],
    references: list[list[float]],
    k: int = 5,
) -> list[Tuple[int, float]]:
    # 쿼리와 레퍼런스 데이터의 임베딩을 계산하고 코사인 유사도를 계산

    sim_scores = [
        (idx, compute_cosine_similarity(query_vec, ref_vec))
        for idx, ref_vec in enumerate(references)
    ]
    sorted_sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[:k]
    top_k_idx = [idx for idx, score in sorted_sim_scores]
    return top_k_idx

refer_dict = extract_json_from(Path("assets/nl_task_db.json"))
input_nl = "cook a egg with critical dependency"
refer_nls = list(refer_dict.keys())

input_nl_vec = embed_data([input_nl], model)
refer_vec = embed_data(refer_nls, model)

top_k_idx = semantic_search(input_nl_vec, refer_vec, k=2)
top_k_dict = {refer_nls[idx]: refer_dict[refer_nls[idx]] for idx in top_k_idx}

for idx, task_info in enumerate(top_k_dict.items(), 1):
    task_nl, task_file_name = task_info
    few_shot_output = extract_json_from(Path(f"assets/tasks/{task_file_name}"))
    few_shot_prompt = f"""
    ### **Example {idx}**
    
    **Input**
    ```
    {task_nl}
    ```
    
    **Output**
    ```json
    {few_shot_output}
    ```
    
    ---
    """
    print(few_shot_prompt)

/home/dongkyu/miniforge3/envs/research/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



    ### **Example 1**
    
    **Input**
    ```
    prepare fried egg and prepare coffee and set the table for lunch
    ```
    
    **Output**
    ```json
    [{'Task': 'Prepare Fried Egg', 'Subtasks': [{'Name': 'Prepare Egg Fry', 'Repetition': 1, 'Type': 'Interaction', 'Executions': {'Objects': {'Egg|-02.04|+00.81|+01.24': 1, 'Pan|+00.72|+00.90|-02.42': 1}, 'PrimitiveActions': ['NAVIGATE_TO Pan|+00.72|+00.90|-02.42', 'GRASP Pan|+00.72|+00.90|-02.42', 'NAVIGATE_TO StoveBurner|-00.04|+00.92|-02.37', 'PLACE_ON_TOP StoveBurner|-00.04|+00.92|-02.37', 'NAVIGATE_TO StoveKnob|-00.02|+00.88|-02.19', 'TOGGLE_ON StoveKnob|-00.02|+00.88|-02.19', 'NAVIGATE_TO Egg|-02.04|+00.81|+01.24', 'GRASP Egg|-02.04|+00.81|+01.24', 'PLACE_ON_TOP Pan|+00.72|+00.90|-02.42', 'SLICE Egg|-02.04|+00.81|+01.24']}, 'Duration': {'Type': 'Controllable', 'Interval': 10}, 'TemporalConstraints': []}, {'Name': 'Turn off stove after cooking', 'Repetition': 1, 'Type': 'Interaction', 'Executions': {'Objects': {'StoveKnob|-